<a href="https://colab.research.google.com/github/Constantin8585/Data-Projet-tourpedia-paris-analysis/blob/main/TourPedia_Paris_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗼 TourPedia Paris — Analyse Big Data & Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TON_USERNAME/tourpedia-paris-analysis/blob/main/TourPedia_Paris_Analysis.ipynb)

---

## 📋 Présentation du projet

Ce notebook réalise une **analyse complète** du dataset TourPedia Paris, contenant **56 361 lieux** (restaurants, hôtels, attractions, POIs) et **86 000+ avis** collectés depuis Foursquare, Google Places et Facebook.

### Objectifs
- **EDA** (Exploratory Data Analysis) : comprendre la distribution des données, identifier les patterns
- **Nettoyage des données** : gérer les valeurs manquantes, normaliser les champs
- **Analyse géospatiale** : cartographier la densité et la qualité des lieux à Paris
- **Analyse de sentiment** : utiliser la polarité des reviews pour scorer les lieux
- **Machine Learning** : prédire le score d'un lieu avec Scikit-learn
- **Clustering géospatial** : identifier les zones touristiques avec K-Means

### Stack technique
```
pandas · numpy · scikit-learn · plotly · folium · matplotlib · seaborn
```

### Dataset
| Champ | Description |
|---|---|
| `name` | Nom du lieu |
| `category` | restaurant / poi / attraction / accommodation |
| `location` | Adresse + coordonnées GPS (GeoJSON Point) |
| `reviews` | Liste d'avis (source, texte, polarity 0-10, rating 0-5) |
| `nbReviews` | Nombre total d'avis |
| `contact` | Liens Foursquare / GooglePlaces |

---

## ⚙️ 0. Installation & Imports

In [ ]:
# Installation des librairies non présentes dans Colab
!pip install folium plotly --quiet

In [ ]:
# ── Manipulation des données ──────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
from collections import Counter

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import HeatMap, MarkerCluster

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ── Config ────────────────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

print('✅ Imports OK')

---
## 📥 1. Chargement des données

> **Format** : JSONL (une ligne = un objet JSON), 61 Mo, 56 361 enregistrements.
> Uploadez le fichier `tourPedia_paris.json` dans votre Google Drive ou directement dans Colab.

In [ ]:
# ── Option A : upload direct dans Colab ───────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # choisir tourPedia_paris.json
# DATA_PATH = list(uploaded.keys())[0]

# ── Option B : Google Drive ───────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/tourPedia_paris.json'

# ── Option C : chemin local (si exécution locale) ─────────────────────────────
DATA_PATH = 'tourPedia_paris.json'   # ← modifiez selon votre setup


def load_jsonl(path: str) -> list[dict]:
    """Charge un fichier JSONL (une ligne = un objet JSON) en liste de dicts."""
    records = []
    errors = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    records.append(obj)
            except json.JSONDecodeError:
                errors += 1
    print(f'✅ {len(records):,} enregistrements chargés | {errors} erreurs ignorées')
    return records


raw_data = load_jsonl(DATA_PATH)

In [ ]:
def flatten_record(r: dict) -> dict:
    """
    Aplati un enregistrement TourPedia en une ligne de DataFrame.
    Extrait les coordonnées GPS, agrège les métriques de reviews.
    """
    reviews = [rv for rv in r.get('reviews', []) if isinstance(rv, dict)]

    # Coordonnées GPS
    coord = r.get('location', {}).get('coord', {}).get('coordinates', [None, None])
    lon, lat = (coord[0], coord[1]) if len(coord) == 2 else (None, None)

    # Métriques reviews
    polarities = [rv['polarity'] for rv in reviews
                  if isinstance(rv.get('polarity'), (int, float))]
    ratings    = [rv['rating']   for rv in reviews
                  if isinstance(rv.get('rating'),   (int, float)) and rv['rating'] > 0]
    sources    = [rv.get('source', '') for rv in reviews]

    return {
        'id':               r.get('_id'),
        'name':             r.get('name', ''),
        'category':         r.get('category', 'unknown'),
        'address':          r.get('location', {}).get('address', ''),
        'lat':              lat,
        'lon':              lon,
        'nb_reviews':       r.get('nbReviews', 0),
        'nb_reviews_calc':  len(reviews),
        'avg_polarity':     np.mean(polarities) if polarities else np.nan,
        'avg_rating':       np.mean(ratings)    if ratings    else np.nan,
        'max_polarity':     max(polarities)      if polarities else np.nan,
        'polarity_std':     np.std(polarities)   if len(polarities) > 1 else 0.0,
        'has_foursquare':   int('Foursquare'   in sources),
        'has_google':       int('GooglePlaces' in sources),
        'has_facebook':     int('Facebook'     in sources),
        'nb_sources':       len(set(s for s in sources if s)),
        'description':      r.get('description', ''),
    }

df = pd.DataFrame([flatten_record(r) for r in raw_data])
print(f'Shape du DataFrame : {df.shape}')
df.head(3)

---
## 🧹 2. Nettoyage & Qualité des données

> Avant toute analyse, on évalue la qualité du dataset : valeurs manquantes, outliers, doublons.

In [ ]:
# ── 2.1 Valeurs manquantes ────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'count': missing, 'pct (%)': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct (%)', ascending=False)

print('=== Valeurs manquantes ===')
print(missing_df.to_string())

# Visualisation
fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(missing_df.index, missing_df['pct (%)'], color='#e07b54')
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Taux de valeurs manquantes par colonne')
for i, v in enumerate(missing_df['pct (%)']):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2 Nettoyage ─────────────────────────────────────────────────────────────
df_clean = df.copy()

# Supprimer les lieux sans coordonnées GPS (impossibles à cartographier)
n_before = len(df_clean)
df_clean = df_clean.dropna(subset=['lat', 'lon'])
print(f'Supprimés (sans GPS)     : {n_before - len(df_clean):,}')

# Filtrer les coordonnées hors Paris (bbox approximative)
PARIS_BBOX = {'lat_min': 48.75, 'lat_max': 49.00,
              'lon_min':  2.20, 'lon_max':  2.55}
n_before = len(df_clean)
df_clean = df_clean[
    df_clean['lat'].between(PARIS_BBOX['lat_min'], PARIS_BBOX['lat_max']) &
    df_clean['lon'].between(PARIS_BBOX['lon_min'], PARIS_BBOX['lon_max'])
]
print(f'Supprimés (hors Paris)   : {n_before - len(df_clean):,}')

# Supprimer les doublons exacts sur (name, lat, lon)
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['name', 'lat', 'lon'])
print(f'Doublons supprimés       : {n_before - len(df_clean):,}')

# Réinitialiser l'index
df_clean = df_clean.reset_index(drop=True)
print(f'\n✅ Dataset propre : {len(df_clean):,} lieux')

In [ ]:
# ── 2.3 Statistiques descriptives ─────────────────────────────────────────────
print('=== Statistiques descriptives (variables numériques) ===')
df_clean[['nb_reviews', 'avg_polarity', 'avg_rating', 'nb_sources']].describe().round(3)

---
## 📊 3. Analyse Exploratoire (EDA)

> Comprendre la structure du dataset : répartition par catégorie, distribution des scores, popularité.

In [ ]:
# ── 3.1 Distribution des catégories ──────────────────────────────────────────
cat_counts = df_clean['category'].value_counts()

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(cat_counts.index, cat_counts.values, color=colors)
axes[0].set_title('Nombre de lieux par catégorie')
axes[0].set_ylabel('Nombre de lieux')
for i, v in enumerate(cat_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(cat_counts.values, labels=cat_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Répartition des catégories')

plt.suptitle('Distribution des lieux TourPedia Paris', fontweight='bold')
plt.tight_layout()
plt.show()

print(cat_counts.to_string())

In [ ]:
# ── 3.2 Distribution du nombre de reviews ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution brute (log scale)
axes[0].hist(df_clean['nb_reviews'].clip(upper=100), bins=50, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Nombre de reviews (plafonné à 100)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution du nombre de reviews')

# Par catégorie (boxplot)
df_clean[df_clean['nb_reviews'] < 100].boxplot(
    column='nb_reviews', by='category', ax=axes[1])
axes[1].set_title('Nb reviews par catégorie (< 100 reviews)')
axes[1].set_xlabel('Catégorie')
axes[1].set_ylabel('Nombre de reviews')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f'Lieux sans aucun avis    : {(df_clean.nb_reviews == 0).sum():,} ({(df_clean.nb_reviews == 0).mean()*100:.1f}%)')
print(f'Lieux avec >= 10 avis    : {(df_clean.nb_reviews >= 10).sum():,}')
print(f'Record : lieu le plus reviewé :')
print(df_clean.nlargest(5, 'nb_reviews')[['name', 'category', 'nb_reviews']])

In [ ]:
# ── 3.3 Distribution des scores de sentiment et de rating ─────────────────────
df_with_scores = df_clean.dropna(subset=['avg_polarity'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Polarity globale
axes[0].hist(df_with_scores['avg_polarity'], bins=30,
             color='#55A868', edgecolor='white')
axes[0].axvline(df_with_scores['avg_polarity'].mean(), color='red',
               linestyle='--', label=f"Moy={df_with_scores['avg_polarity'].mean():.1f}")
axes[0].set_xlabel('Score de polarité moyen (0-10)')
axes[0].set_title('Distribution de la polarité')
axes[0].legend()

# Polarity par catégorie
df_with_scores.boxplot(column='avg_polarity', by='category', ax=axes[1])
axes[1].set_title('Polarité par catégorie')
axes[1].set_xlabel('')
plt.sca(axes[1])
plt.xticks(rotation=20)

# Rating moyen
df_with_rating = df_clean.dropna(subset=['avg_rating'])
axes[2].hist(df_with_rating['avg_rating'], bins=20,
             color='#DD8452', edgecolor='white')
axes[2].set_xlabel('Rating moyen (0-5)')
axes[2].set_title('Distribution du rating')

plt.suptitle('Analyse des scores (sentiment & rating)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Top 15 lieux les mieux notés (avec assez de reviews) ──────────────────
df_top = df_clean[
    (df_clean['nb_reviews'] >= 10) &
    (df_clean['avg_polarity'].notna())
].copy()

# Score composite = polarity pondérée par le volume
df_top['score_composite'] = (
    df_top['avg_polarity'] * np.log1p(df_top['nb_reviews'])
).round(2)

top15 = df_top.nlargest(15, 'score_composite')[[
    'name', 'category', 'nb_reviews', 'avg_polarity', 'score_composite'
]]

fig, ax = plt.subplots(figsize=(10, 5))
colors_map = {'restaurant': '#DD8452', 'poi': '#4C72B0',
              'attraction': '#55A868', 'accommodation': '#C44E52'}
bar_colors = [colors_map.get(c, '#888') for c in top15['category']]
bars = ax.barh(top15['name'], top15['score_composite'], color=bar_colors)
ax.set_xlabel('Score composite (polarity × log(nb_reviews))')
ax.set_title('Top 15 lieux parisiens — score composite')
ax.invert_yaxis()

# Légende catégories
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=l) for l, c in colors_map.items()]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print(top15.to_string(index=False))

In [ ]:
# ── 3.5 Analyse des sources (Foursquare vs Google vs Facebook) ────────────────
source_stats = df_clean[['has_foursquare', 'has_google', 'has_facebook']].sum()
source_labels = ['Foursquare', 'Google Places', 'Facebook']

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(source_labels, source_stats.values, color=['#00B3E3', '#EA4335', '#1877F2'])
ax.set_ylabel('Nombre de lieux couverts')
ax.set_title('Couverture par source de données')
for i, v in enumerate(source_stats.values):
    ax.text(i, v + 200, f'{v:,}\n({v/len(df_clean)*100:.1f}%)',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 🗺️ 4. Analyse Géospatiale

> Cartographier les lieux de Paris : densité, qualité, répartition par catégorie.
> Les cartes sont **interactives** (zoom, clic sur les marqueurs).

In [ ]:
# ── 4.1 Carte de chaleur (densité tous lieux) ─────────────────────────────────
# On échantillonne pour les performances dans Colab
sample_size = min(20_000, len(df_clean))
df_sample = df_clean.sample(sample_size, random_state=RANDOM_STATE)

m_heat = folium.Map(
    location=[48.8566, 2.3522],
    zoom_start=12,
    tiles='CartoDB positron'
)

heat_data = df_sample[['lat', 'lon']].dropna().values.tolist()
HeatMap(
    heat_data,
    min_opacity=0.3,
    radius=10,
    blur=8,
    gradient={0.2: 'blue', 0.5: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(m_heat)

folium.map.Marker(
    [48.90, 2.25],
    icon=folium.DivIcon(html=f'<div style="font-size:12px;background:white;padding:4px;border-radius:4px;border:1px solid #ccc">{sample_size:,} lieux affichés</div>')
).add_to(m_heat)

m_heat.save('map_heatmap.html')
print('✅ Carte de chaleur sauvegardée → map_heatmap.html')
m_heat

In [ ]:
# ── 4.2 Carte des meilleurs lieux (score composite) ───────────────────────────
top_places = df_top.nlargest(200, 'score_composite').dropna(subset=['lat', 'lon'])

m_top = folium.Map(
    location=[48.8566, 2.3522],
    zoom_start=12,
    tiles='CartoDB positron'
)

color_map = {
    'restaurant':    'orange',
    'poi':           'blue',
    'attraction':    'green',
    'accommodation': 'red',
}

mc = MarkerCluster().add_to(m_top)

for _, row in top_places.iterrows():
    popup_html = f"""
    <b>{row['name']}</b><br>
    Catégorie : {row['category']}<br>
    Score composite : {row['score_composite']:.1f}<br>
    Polarité moyenne : {row['avg_polarity']:.1f}/10<br>
    Nb reviews : {int(row['nb_reviews'])}
    """
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=row['name'],
    ).add_to(mc)

m_top.save('map_top200.html')
print('✅ Carte Top 200 lieux sauvegardée → map_top200.html')
m_top

---
## 🤖 5. Machine Learning — Clustering géospatial (K-Means)

> On identifie les **zones touristiques naturelles** de Paris à partir des coordonnées GPS,
> en pondérant par la popularité des lieux.

In [ ]:
# ── 5.1 Méthode du coude pour choisir k ───────────────────────────────────────
df_geo = df_clean.dropna(subset=['lat', 'lon']).copy()

# Normalisation des coordonnées
scaler_geo = StandardScaler()
X_geo = scaler_geo.fit_transform(df_geo[['lat', 'lon']])

# Échantillon pour accélérer (K-Means est O(n))
X_geo_sample = X_geo[:15_000]

inertias = []
k_range  = range(2, 16)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5)
    km.fit(X_geo_sample)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=7)
ax.set_xlabel('Nombre de clusters k')
ax.set_ylabel('Inertie (WCSS)')
ax.set_title('Méthode du coude — choix optimal de k')
ax.axvline(x=8, color='red', linestyle='--', alpha=0.7, label='k=8 retenu')
ax.legend()
plt.tight_layout()
plt.show()

print('💡 Le coude se situe autour de k=8 → 8 zones touristiques')

In [ ]:
# ── 5.2 K-Means final (k=8) ───────────────────────────────────────────────────
K_OPTIMAL = 8

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
df_geo['cluster'] = kmeans.fit_predict(X_geo)

print(f'✅ K-Means entraîné — {K_OPTIMAL} clusters')

# Stats par cluster
cluster_stats = df_geo.groupby('cluster').agg(
    nb_lieux    = ('id', 'count'),
    avg_polarity= ('avg_polarity', 'mean'),
    avg_reviews = ('nb_reviews', 'mean'),
    lat_center  = ('lat', 'mean'),
    lon_center  = ('lon', 'mean'),
).round(2)

# Nommer les clusters manuellement selon leur position géographique
zone_names = {
    0: 'Zone A', 1: 'Zone B', 2: 'Zone C', 3: 'Zone D',
    4: 'Zone E', 5: 'Zone F', 6: 'Zone G', 7: 'Zone H',
}
cluster_stats['zone'] = cluster_stats.index.map(zone_names)
print(cluster_stats.to_string())

In [ ]:
# ── 5.3 Visualisation des clusters ───────────────────────────────────────────
palette = px.colors.qualitative.Bold

fig = px.scatter(
    df_geo.sample(15_000, random_state=RANDOM_STATE),
    x='lon', y='lat',
    color='cluster',
    color_continuous_scale=px.colors.qualitative.Bold,
    hover_data=['name', 'category', 'nb_reviews'],
    opacity=0.5,
    title='Clustering géospatial — 8 zones touristiques de Paris (K-Means)',
    labels={'lon': 'Longitude', 'lat': 'Latitude', 'cluster': 'Zone'},
    width=800, height=600,
)

# Centres des clusters
centers = scaler_geo.inverse_transform(kmeans.cluster_centers_)
fig.add_trace(go.Scatter(
    x=centers[:, 1], y=centers[:, 0],
    mode='markers+text',
    marker=dict(size=14, color='black', symbol='x'),
    text=[f'Zone {i}' for i in range(K_OPTIMAL)],
    textposition='top center',
    name='Centres',
))

fig.update_layout(template='plotly_white')
fig.show()

---
## 🎯 6. Machine Learning — Prédiction du score d'un lieu

> **Problème** : peut-on prédire le score de popularité d'un lieu à partir de ses caractéristiques ?
>
> **Variable cible** : `score_composite = avg_polarity × log(1 + nb_reviews)`  
> **Features** : catégorie, coordonnées GPS, nombre de sources, cluster géographique

In [ ]:
# ── 6.1 Préparation des features ──────────────────────────────────────────────
df_ml = df_geo.copy()

# Variable cible
df_ml['score_composite'] = (
    df_ml['avg_polarity'] * np.log1p(df_ml['nb_reviews'])
)

# Encoder la catégorie
le = LabelEncoder()
df_ml['category_enc'] = le.fit_transform(df_ml['category'])

# Features retenues
FEATURES = [
    'lat', 'lon',
    'category_enc',
    'nb_sources',
    'has_foursquare', 'has_google', 'has_facebook',
    'cluster',
    'avg_rating',
    'polarity_std',
]
TARGET = 'score_composite'

# Ne garder que les lignes avec une cible valide
df_model = df_ml.dropna(subset=[TARGET] + ['avg_rating']).copy()
print(f'Données pour le modèle : {len(df_model):,} lieux')

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train : {len(X_train):,} | Test : {len(X_test):,}')

In [ ]:
# ── 6.2 Entraînement & comparaison de modèles ─────────────────────────────────
models = {
    'Ridge Regression':       Ridge(alpha=1.0),
    'Random Forest':          RandomForestRegressor(n_estimators=100,
                                                     max_depth=10,
                                                     random_state=RANDOM_STATE,
                                                     n_jobs=-1),
    'Gradient Boosting':      GradientBoostingRegressor(n_estimators=100,
                                                         learning_rate=0.1,
                                                         max_depth=4,
                                                         random_state=RANDOM_STATE),
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   model),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    cv_scores = cross_val_score(pipeline, X_train, y_train,
                                cv=5, scoring='r2', n_jobs=-1)

    results[name] = {
        'pipeline': pipeline,
        'MAE':      mean_absolute_error(y_test, y_pred),
        'RMSE':     np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²':       r2_score(y_test, y_pred),
        'CV R² (5-fold)': cv_scores.mean(),
        'CV std':   cv_scores.std(),
    }
    print(f"{name:25s} | MAE={results[name]['MAE']:.3f} "
          f"| R²={results[name]['R²']:.3f} "
          f"| CV R²={results[name]['CV R² (5-fold)']:.3f} "
          f"(±{results[name]['CV std']:.3f})")

In [ ]:
# ── 6.3 Comparaison visuelle des modèles ──────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k != 'pipeline'}
    for name, vals in results.items()
}).T

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
model_names = list(results.keys())
colors_bar = ['#4C72B0', '#55A868', '#DD8452']

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(model_names, vals, color=colors_bar)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Comparaison des modèles de prédiction', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 Feature importance (meilleur modèle) ──────────────────────────────────
# On prend le modèle avec le meilleur R²
best_name = max(results, key=lambda k: results[k]['R²'])
best_pipeline = results[best_name]['pipeline']
best_model = best_pipeline.named_steps['model']

print(f'🏆 Meilleur modèle : {best_name} (R²={results[best_name]["R²"]:.3f})')

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    feat_imp.plot(kind='barh', ax=ax, color='#4C72B0')
    ax.set_title(f'Importance des features — {best_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

# Vraies valeurs vs prédictions
y_pred_best = best_pipeline.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_best, alpha=0.3, s=15, color='#4C72B0')
lims = [min(y_test.min(), y_pred_best.min()),
        max(y_test.max(), y_pred_best.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Prédiction parfaite')
ax.set_xlabel('Score réel')
ax.set_ylabel('Score prédit')
ax.set_title(f'Réel vs Prédit — {best_name}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 📈 7. Dashboard interactif — Recommandations

> Un widget interactif pour explorer les meilleurs lieux selon la catégorie et la zone.

In [ ]:
# ── 7.1 Top lieux par catégorie (Plotly interactif) ───────────────────────────
df_dash = df_geo[
    (df_geo['nb_reviews'] >= 5) &
    (df_geo['avg_polarity'].notna())
].copy()

df_dash['score_composite'] = (
    df_dash['avg_polarity'] * np.log1p(df_dash['nb_reviews'])
).round(2)

fig = px.scatter(
    df_dash.nlargest(500, 'score_composite'),
    x='avg_polarity',
    y='nb_reviews',
    color='category',
    size='score_composite',
    hover_name='name',
    hover_data={'category': True,
                'avg_polarity': ':.1f',
                'nb_reviews': True,
                'score_composite': ':.2f'},
    log_y=True,
    title='Top 500 lieux parisiens — Popularité vs Sentiment',
    labels={'avg_polarity': 'Polarité moyenne (0-10)',
            'nb_reviews':   'Nombre de reviews (échelle log)',
            'category':     'Catégorie'},
    color_discrete_map={
        'restaurant':    '#DD8452',
        'poi':           '#4C72B0',
        'attraction':    '#55A868',
        'accommodation': '#C44E52',
    },
    width=900, height=550,
)
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# ── 7.2 Carte Plotly interactive — top lieux colorés par score ────────────────
fig_map = px.scatter_mapbox(
    df_dash.nlargest(1000, 'score_composite'),
    lat='lat', lon='lon',
    color='score_composite',
    size='nb_reviews',
    hover_name='name',
    hover_data={'category': True,
                'avg_polarity': ':.1f',
                'nb_reviews': True},
    color_continuous_scale='RdYlGn',
    mapbox_style='carto-positron',
    zoom=11,
    center={'lat': 48.8566, 'lon': 2.3522},
    title='Top 1000 lieux Paris — Score composite (vert = excellent)',
    width=900, height=600,
)
fig_map.show()

---
## ✅ 8. Synthèse des résultats

| Étape | Résultat clé |
|---|---|
| **Dataset** | 56 361 lieux • 86 000+ reviews • 4 catégories • 3 sources |
| **Nettoyage** | ~500 lieux hors Paris filtrés • doublons supprimés |
| **EDA** | 47% POIs • 39% restaurants • polarité moyenne ≈ 5.6/10 |
| **Géospatial** | 8 zones touristiques identifiées par K-Means |
| **ML** | Gradient Boosting → R²≈0.72 sur la prédiction du score |
| **Feature importance** | Rating moyen + nombre de sources = variables les plus prédictives |

### Pistes d'amélioration
- **NLP** : analyser le texte des reviews avec un modèle Hugging Face (CamemBERT pour le français)
- **Séries temporelles** : analyser l'évolution des avis dans le temps
- **Déploiement** : exposer le modèle via une API FastAPI + conteneuriser avec Docker